# Proactive Agents: Noise Reduction

**Level:** Advanced · **Time:** 90 min

In this notebook, we simulate the transition from a spammy, reactive script to a polite, intelligent proactive agent.

We will cover 4 distinct patterns:
1. **The Spam Anti-Pattern:** Alerting on every single event.
2. **Signature Deduplication:** Dropping duplicate events using a simulated Redis cache.
3. **Hysteresis Cooldowns:** Preventing flapping when metrics oscillate.
4. **Quiet Hours Routing:** Downgrading notifications based on time and urgency.

---
## Pattern 1: The Spam Anti-Pattern

A naive agent alerts on every single error it sees in the stream.

In [ ]:
def spam_agent(error_stream):
    print("Starting stream processing...")
    for error in error_stream:
        print(f"🚨 [Agent] Sending Slack Alert: {error}")

stream = ["DB Timeout", "DB Timeout", "DB Timeout", "DB Timeout"]
spam_agent(stream)
print("❌ [Result] The engineer's phone buzzed 4 times in 1 second. They muted the channel.")


---
## Pattern 2: Signature Deduplication

By hashing the error and checking a cache, we ensure the agent only alerts once per unique issue.

In [ ]:
import hashlib

mock_redis_cache = set()

def get_signature(error_msg):
    return hashlib.md5(error_msg.encode()).hexdigest()

def deduplicating_agent(error_stream):
    print("Starting stream processing with Deduplication...")
    for error in error_stream:
        sig = get_signature(error)
        
        if sig in mock_redis_cache:
            print(f"🛡️ [Agent] Suppressing duplicate error: {error}")
            continue
            
        print(f"🚨 [Agent] Sending Slack Alert: {error}")
        mock_redis_cache.add(sig)

stream = ["DB Timeout", "DB Timeout", "UI Glitch", "DB Timeout"]
deduplicating_agent(stream)


---
## Pattern 3: Hysteresis Cooldowns

When monitoring continuous metrics, we must use an Activation threshold AND a Recovery threshold to prevent flapping.

In [ ]:
class HysteresisMonitor:
    def __init__(self):
        self.state = "NORMAL"
        self.activation_threshold = 90
        self.recovery_threshold = 70
        
    def process_metric(self, cpu_usage):
        print(f"\nMetric: {cpu_usage}% CPU | Current State: {self.state}")
        
        if self.state == "NORMAL":
            if cpu_usage >= self.activation_threshold:
                print("🚨 [Agent] Threshold breached! Sending Alert. Entering COOLDOWN.")
                self.state = "COOLDOWN"
        
        elif self.state == "COOLDOWN":
            if cpu_usage <= self.recovery_threshold:
                print("✅ [Agent] Metric fully recovered. Returning to NORMAL.")
                self.state = "NORMAL"
            else:
                print("🛡️ [Agent] Metric fluctuating. Suppressing alert due to Hysteresis.")

monitor = HysteresisMonitor()
cpu_stream = [85, 91, 89, 92, 65, 95]

for metric in cpu_stream:
    monitor.process_metric(metric)


---
## Pattern 4: Quiet Hours Routing

Respecting user consent by intercepting non-critical alerts at 3 AM and downgrading them to a digest.

In [ ]:
def notification_router(alert_urgency, current_hour):
    print(f"\n[Router] Received {alert_urgency} alert at {current_hour}:00")
    
    is_quiet_hours = current_hour < 8 or current_hour >= 20
    
    if alert_urgency == "P1_CRITICAL":
        print("🚨 [Router] CRITICAL. Paging engineer immediately regardless of time.")
        return
        
    if is_quiet_hours:
        print("📥 [Router] Quiet Hours active. Downgrading alert to Daily Digest Email.")
    else:
        print("💬 [Router] Working hours active. Sending Slack push notification.")

# Scenario A: Minor bug at 3 AM
notification_router("P4_MINOR", 3)

# Scenario B: Minor bug at 2 PM
notification_router("P4_MINOR", 14)

# Scenario C: Database down at 4 AM
notification_router("P1_CRITICAL", 4)
